# PayGuard — Step 2: Schema Adaptation & Fraud Injection

Goal of this notebook:
1. Rename PaySim's columns to a UPI-style schema (VPAs, transaction mode, timestamps)
2. Inject additional synthetic fraud patterns (off-hour spikes, new-receiver anomalies) on top of PaySim's original fraud labels
3. Save the final adapted dataset that feature engineering (notebook 3) will build on

Input: `../data/paysim_sample.csv` (80k rows, from notebook 1)

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

df = pd.read_csv("../data/paysim_sample.csv")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded: 80,000 rows x 11 columns


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,211,PAYMENT,3781.26,C1337578744,3031.00,0.00,M681498494,0.00,0.00,0,0
1,249,CASH_OUT,8992.28,C1928497485,0.00,0.00,C1567947232,480316.92,489309.20,0,0
2,18,CASH_IN,138752.43,C1752665596,12050.00,150802.43,C1607054435,1873749.18,2075917.94,0,0
3,11,PAYMENT,11798.96,C2024985793,66675.72,54876.77,M921723804,0.00,0.00,0,0
4,39,CASH_IN,244953.00,C1053187051,2295919.90,2540872.90,C1940729177,3808595.55,3563642.56,0,0


## 1. Convert `step` into a real timestamp

PaySim's `step` is "1 step = 1 hour" starting from some arbitrary point. We anchor it to a real start date so we get genuine timestamps — which matters for the time-of-day/off-hour behavioral feature later.

In [3]:
START_DATE = datetime(2026, 1, 1, 0, 0, 0)

df["timestamp"] = df["step"].apply(lambda h: START_DATE + timedelta(hours=int(h)))
df[["step", "timestamp"]].head()

,step,timestamp
0,211,2026-01-09 19:00:00
1,249,2026-01-11 09:00:00
2,18,2026-01-01 18:00:00
3,11,2026-01-01 11:00:00
4,39,2026-01-02 15:00:00


## 2. Generate UPI-style VPAs

PaySim's `nameOrig`/`nameDest` use a prefix convention: `C...` = customer account, `M...` = merchant account.
We turn these into VPA-looking strings and capture whether a transaction is P2P (customer→customer) or P2M (customer→merchant) — a useful feature PaySim's raw `type` column doesn't fully capture on its own.

In [4]:
BANK_HANDLES = ["oksbi", "okicici", "okaxis", "okhdfcbank", "ybl", "paytm"]

def to_vpa(account_id: str) -> str:
    """Turn a PaySim account id like 'C1231006815' or 'M1979787155' into a UPI-style VPA."""
    is_merchant = account_id.startswith("M")
    numeric_part = account_id[1:]
    handle = np.random.choice(BANK_HANDLES)
    prefix = "merchant" if is_merchant else "user"
    return f"{prefix}{numeric_part[-8:]}@{handle}"

# deterministic per unique account so the same account always maps to the same VPA
unique_accounts = pd.concat([df["nameOrig"], df["nameDest"]]).unique()
vpa_map = {acc: to_vpa(acc) for acc in unique_accounts}

df["sender_vpa"] = df["nameOrig"].map(vpa_map)
df["receiver_vpa"] = df["nameDest"].map(vpa_map)
df["receiver_is_merchant"] = df["nameDest"].str.startswith("M").astype(int)

df[["nameOrig", "sender_vpa", "nameDest", "receiver_vpa", "receiver_is_merchant"]].head()

,nameOrig,sender_vpa,nameDest,receiver_vpa,receiver_is_merchant
0,C1337578744,user37578744@okhdfcbank,M681498494,merchant81498494@okhdfcbank,1
1,C1928497485,user28497485@ybl,C1567947232,user67947232@okicici,0
2,C1752665596,user52665596@okaxis,C1607054435,user07054435@paytm,0
3,C2024985793,user24985793@ybl,M921723804,merchant21723804@okaxis,1
4,C1053187051,user53187051@ybl,C1940729177,user40729177@okicici,0


## 3. Rename remaining columns to the UPI schema

Keeping `isFlaggedFraud` (PaySim's own naive rule) as a reference baseline — useful later to show our ML approach beats a simple threshold rule, but it is **not** used as a model feature.

In [5]:
df = df.rename(columns={
    "type": "transaction_mode",
    "amount": "amount",
    "oldbalanceOrg": "sender_balance_before",
    "newbalanceOrig": "sender_balance_after",
    "oldbalanceDest": "receiver_balance_before",
    "newbalanceDest": "receiver_balance_after",
    "isFraud": "is_fraud",
    "isFlaggedFraud": "is_flagged_fraud_baseline",
})

upi_df = df[[
    "timestamp", "sender_vpa", "receiver_vpa", "receiver_is_merchant",
    "transaction_mode", "amount",
    "sender_balance_before", "sender_balance_after",
    "receiver_balance_before", "receiver_balance_after",
    "is_fraud", "is_flagged_fraud_baseline",
]].copy()

upi_df["fraud_source"] = np.where(upi_df["is_fraud"] == 1, "paysim_original", "none")

print(upi_df.shape)
upi_df.head()

(80000, 13)


,timestamp,sender_vpa,receiver_vpa,receiver_is_merchant,transaction_mode,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,is_fraud,is_flagged_fraud_baseline,fraud_source
0,2026-01-09 19:00:00,user37578744@okhdfcbank,merchant81498494@okhdfcbank,1,PAYMENT,3781.26,3031.00,0.00,0.00,0.00,0,0,none
1,2026-01-11 09:00:00,user28497485@ybl,user67947232@okicici,0,CASH_OUT,8992.28,0.00,0.00,480316.92,489309.20,0,0,none
2,2026-01-01 18:00:00,user52665596@okaxis,user07054435@paytm,0,CASH_IN,138752.43,12050.00,150802.43,1873749.18,2075917.94,0,0,none
3,2026-01-01 11:00:00,user24985793@ybl,merchant21723804@okaxis,1,PAYMENT,11798.96,66675.72,54876.77,0.00,0.00,0,0,none
4,2026-01-02 15:00:00,user53187051@ybl,user40729177@okicici,0,CASH_IN,244953.00,2295919.90,2540872.90,3808595.55,3563642.56,0,0,none


## 4. Inject additional synthetic fraud patterns

PaySim's own fraud only covers one behavior (large transfers immediately cashed out). We add two more UPI-realistic patterns on top, each with **randomized parameters** so the model can't just learn a single fixed threshold:

1. **Off-hour spike fraud** — a transaction shifted into unusual hours (randomly between 1–4 AM) with an inflated amount
2. **New-receiver anomaly fraud** — a sender's first-ever transaction to a brand-new receiver, with a moderately high amount

We inject these by taking a small number of existing **non-fraud** rows and transforming them, rather than fabricating rows from nothing — this keeps the injected transactions grounded in realistic balance/amount relationships from the underlying simulator.

In [6]:
INJECTION_FRACTION = 0.0015  # ~0.15% of the dataset gets injected fraud, on top of PaySim's own ~0.13%
n_inject_total = int(len(upi_df) * INJECTION_FRACTION)
n_offhour = n_inject_total // 2
n_new_receiver = n_inject_total - n_offhour

non_fraud_idx = upi_df[upi_df["is_fraud"] == 0].index
injection_idx = np.random.choice(non_fraud_idx, size=n_inject_total, replace=False)
offhour_idx = injection_idx[:n_offhour]
new_receiver_idx = injection_idx[n_offhour:]

print(f"Injecting {n_offhour} off-hour fraud cases and {n_new_receiver} new-receiver fraud cases")

Injecting 60 off-hour fraud cases and 60 new-receiver fraud cases


In [7]:
# --- Pattern 1: off-hour spike fraud ---
for idx in offhour_idx:
    random_hour = np.random.randint(1, 5)  # 1 AM - 4 AM, varied each time
    ts = upi_df.at[idx, "timestamp"]
    upi_df.at[idx, "timestamp"] = ts.replace(hour=random_hour, minute=np.random.randint(0, 60))

    amount_multiplier = np.random.uniform(2.5, 6.0)  # varied inflation, not a fixed rule
    upi_df.at[idx, "amount"] = round(upi_df.at[idx, "amount"] * amount_multiplier, 2)
    upi_df.at[idx, "is_fraud"] = 1
    upi_df.at[idx, "fraud_source"] = "injected_offhour"

print("Off-hour injection done.")

Off-hour injection done.


In [8]:
# --- Pattern 2: new-receiver anomaly fraud ---
# Give the sender a receiver VPA they have never transacted with before in this dataset,
# and bump the amount moderately (not as extreme as off-hour, to keep patterns distinguishable
# but still overlapping with normal amounts so it's not trivially separable).
all_receivers = upi_df["receiver_vpa"].unique()

for idx in new_receiver_idx:
    sender = upi_df.at[idx, "sender_vpa"]
    known_receivers = set(upi_df.loc[upi_df["sender_vpa"] == sender, "receiver_vpa"])

    candidate_pool = [r for r in all_receivers if r not in known_receivers]
    new_receiver = np.random.choice(candidate_pool) if candidate_pool else upi_df.at[idx, "receiver_vpa"]

    upi_df.at[idx, "receiver_vpa"] = new_receiver
    amount_multiplier = np.random.uniform(1.3, 2.8)  # milder, overlapping range
    upi_df.at[idx, "amount"] = round(upi_df.at[idx, "amount"] * amount_multiplier, 2)
    upi_df.at[idx, "is_fraud"] = 1
    upi_df.at[idx, "fraud_source"] = "injected_new_receiver"

print("New-receiver injection done.")

New-receiver injection done.


## 5. Check the result

Confirm the new fraud rate and the breakdown by source (original PaySim fraud vs. each injected pattern).

In [9]:
print("Fraud rate after injection:", f"{upi_df['is_fraud'].mean() * 100:.4f}%")
print()
print("Breakdown by fraud source:")
print(upi_df["fraud_source"].value_counts())

Fraud rate after injection: 0.2787%

Breakdown by fraud source:
fraud_source
none                     79777
paysim_original            103
injected_offhour            60
injected_new_receiver       60
Name: count, dtype: int64


## 6. Save the adapted dataset

In [10]:
upi_df = upi_df.sort_values("timestamp").reset_index(drop=True)
upi_df.to_csv("../data/paysim_upi_adapted.csv", index=False)
print(f"Saved {upi_df.shape[0]:,} rows x {upi_df.shape[1]} columns to ../data/paysim_upi_adapted.csv")
upi_df.head()

Saved 80,000 rows x 13 columns to ../data/paysim_upi_adapted.csv


,timestamp,sender_vpa,receiver_vpa,receiver_is_merchant,transaction_mode,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,is_fraud,is_flagged_fraud_baseline,fraud_source
0,2026-01-01 01:00:00,user67042315@paytm,user86084959@okhdfcbank,0,CASH_OUT,607616.73,0.0,0.00,1359923.49,2107778.11,0,0,none
1,2026-01-01 01:00:00,user38933223@okaxis,merchant74897456@okaxis,1,PAYMENT,16859.52,0.0,0.00,0.00,0.00,0,0,none
2,2026-01-01 01:00:00,user42891691@okaxis,user92292416@ybl,0,CASH_OUT,45756.23,0.0,0.00,1393202.52,3420103.09,0,0,none
3,2026-01-01 01:00:00,user94748968@ybl,merchant44815218@paytm,1,PAYMENT,4881.52,0.0,0.00,0.00,0.00,0,0,none
4,2026-01-01 01:00:00,user20305914@okicici,user57863579@okicici,0,CASH_IN,104605.78,4968145.5,5072751.28,207302.14,92307.65,0,0,none


## 7. Data dictionary (for your report)

| Column | Description |
|---|---|
| `timestamp` | Transaction time, derived from PaySim's `step` anchored to a real start date |
| `sender_vpa` | Synthetic UPI VPA for the sender |
| `receiver_vpa` | Synthetic UPI VPA for the receiver |
| `receiver_is_merchant` | 1 if receiver is a merchant account (P2M), 0 if peer (P2P) |
| `transaction_mode` | Original PaySim transaction type (CASH_OUT, PAYMENT, CASH_IN, TRANSFER, DEBIT) |
| `amount` | Transaction amount |
| `sender_balance_before` / `sender_balance_after` | Sender's account balance before/after the transaction |
| `receiver_balance_before` / `receiver_balance_after` | Receiver's account balance before/after the transaction |
| `is_fraud` | Final label: 1 if fraudulent (original PaySim fraud OR injected pattern) |
| `is_flagged_fraud_baseline` | PaySim's own naive rule flag — reference only, not used as a model feature |
| `fraud_source` | `paysim_original`, `injected_offhour`, `injected_new_receiver`, or `none` — kept for our own analysis, **drop before training** so models don't use it as a shortcut label |

**Important:** `fraud_source` must be excluded from the feature set in notebook 3 — it directly encodes the label and would leak information.

## Next notebook: `03_feature_engineering_and_preprocessing.ipynb`

- Engineer behavioral features: transaction velocity, deviation from sender's historical average, new-receiver flag, time-of-day anomaly score, balance ratio
- Stratified train/test split
- Build the imbalance-handling comparison (SMOTE vs. undersampling vs. class-weighting)